In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import Counter

Impurity of a node in a decision tree can be measured using:  
* Entropy   
   High Entopy is high impurity and low entropy means low impurity.  
   Entropy= $∑_{i=1}^{c}$ - $p_{i}$ $log_{2}$ ($p_{i}$)
     
* Gini  
   The probablity that randomly chosen instance can be misclassified.  
   High Gini is high impurity and low Gini means low impurity.   
   Gini= 1 - $∑_{i=1}^{c}$ - $p_{i}^{2}$
       
* Information Gain  
Substraction of child node impurity with parent node impurity.  
IG = Impurity(parent) - $∑_{i=1}^{k}$ $w_{i}$ Impurity($child_{i}$)  
where,  
$w_{i}$= weighted average of impurities.    

In [2]:
data=pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\Machine Learning\0.Data\binomial_logistic_regression_dataset.csv')
data.head()

,Hours_Studied,Attendance_Percentage,Pass(1)_Fail(0)
0,5.993428,66.710050,1
1,4.723471,69.398190,1
2,6.295377,82.472936,1
3,8.046060,81.103703,1
4,4.531693,74.790984,1


In [3]:
data.rename(columns={'Pass(1)_Fail(0)':'Pass'},inplace=True)

In [4]:
x=data.drop(columns='Pass').values
y=data['Pass'].values

In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [6]:
class Node:
    def __init__(self,feature_idx=None,thresold=None,info_gain=None,left=None,right=None,value=None):
        #descision node
        self.feature_idx=feature_idx
        self.thresold=thresold
        self.info_gain=info_gain
        self.left=left
        self.right=right
        #leaf node
        self.value=value
        

In [11]:
class DecisionTree:
    def __init__(self,min_sample_split=2,max_depth=2):
        self.min_sample_split=min_sample_split
        self.max_depth=max_depth

    def build(self,dataset,curr_depth=0):
        x,y=dataset[:,:-1],dataset[:,-1]
        n_samples,n_features=x.shape

        if n_samples>=self.min_sample_split and curr_depth < self.max_depth:
            best_split=self.best_split(dataset,n_features)

            if best_split['info_gain']>0:
                left_node=self.build(best_split['left_dataset'],curr_depth+1)
                right_node=self.build(best_split['right_dataset'],curr_depth+1)

                return Node(best_split['feature_idx'],best_split['thresold'],best_split['info_gain'],left_node,right_node)
        
        leaf=Counter(y).most_common(1)[0][0]
        return Node(value=leaf)
    
    def best_split(self,dataset,n_features):
        best_split={'feature_idx':None, 'thresold':None, 'info_gain':-1, 'left_dataset':None, 'right_dataset':None, 'value':None}

        for i in range(n_features):
            feature_val=dataset[:,i]
            thresolds=np.unique(feature_val)

            for thresold in thresolds:
                left_dataset,right_dataset=self.split(dataset, i, thresold)

                if len(left_dataset) and len(right_dataset):
                    parent_y,left_y,right_y=dataset[:,-1],left_dataset[:,-1],right_dataset[:,-1]

                    info_gain= self.information_gain(parent_y,left_y,right_y)

                    if info_gain>best_split['info_gain']:
                        best_split['feature_idx']=i
                        best_split['thresold']=thresold
                        best_split['info_gain']=info_gain
                        best_split['left_dataset']=left_dataset
                        best_split['right_dataset']=right_dataset
        
        return best_split
    
    def split(self,dataset,feature_idx,thresold):
        left_dataset=np.array([row for row in dataset if row[feature_idx]<=thresold])
        right_dataset=np.array([row for row in dataset if row[feature_idx]>thresold])

        return left_dataset,right_dataset

    def information_gain(self,parent_y,left_y,right_y):
        left_weight=len(left_y)/len(parent_y)
        right_weight=len(right_y)/len(parent_y)
        
        information_gain=self.entropy(parent_y)-(left_weight * self.entropy(left_y) + right_weight * self.entropy(right_y))

        return information_gain
    
    def entropy(self,y):
        entropy=0
        class_labels=np.unique(y)
        for class_label in class_labels:
            p = len(y[y == class_label]) / len(y)

            entropy+= - p * np.log2(p)

        return entropy
    
    def fit(self,x,y):
        dataset=np.concatenate([x,y.reshape(-1,1)],axis=1)
        self.root=self.build(dataset)

    def predict(self,x):
        predictions=[self.predict_class(row,self.root) for row in x]
        return predictions
    
    def predict_class(self,row,node):
        if node.value is not None:
            return node.value
        
        feature_val=row[node.feature_idx]
        if feature_val<=node.thresold:
            return self.predict_class(row,node.left)
        else:
            return self.predict_class(row,node.right)

In [12]:
model=DecisionTree()
model.fit(x_train,y_train)
predictions=model.predict(x_test)
accuracy=np.mean(predictions==y_test)
accuracy*100

82.66666666666667